# SIMULACRO DE CONTROL: Análisis de Datos en Salud (GRD y APS)

**Instrucciones:**
Estás analizando una base de datos de un grupo de hospitales chilenos. El dataset contiene información clínica, de estadía y costos asociados a distintos pacientes. Sin embargo, como suele ocurrir en la vida real, **los datos no están perfectamente limpios**.

Tu objetivo es limpiar, explorar y responder preguntas clave utilizando `pandas`, `matplotlib` y `seaborn`.

---

## 1. Carga y Preparación del Entorno

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# ---------------------------------------------------------
# NO MODIFICAR ESTA SECCIÓN - GENERACIÓN DE DATOS SUCIOS
# ---------------------------------------------------------
np.random.seed(123)
n_pacientes = 500

datos = {
    'ID_Paciente': range(1, n_pacientes + 1),
    'Edad': np.random.normal(55, 18, n_pacientes),
    'Sexo': np.random.choice(['F', 'M'], n_pacientes, p=[0.52, 0.48]),
    'IMC': np.random.normal(28, 6, n_pacientes),
    'Presion_Sistolica': np.random.normal(130, 20, n_pacientes),
    'Dias_Estada': np.random.exponential(scale=5, size=n_pacientes),
    'Costo_Atencion_CLP': np.random.normal(500000, 150000, n_pacientes)
}

df_hospital = pd.DataFrame(datos)

# Introduciendo 'suciedad' y lógica de negocio
# Las edades negativas las pasamos a NaN (esto convierte la columna a float automáticamente)
df_hospital.loc[df_hospital['Edad'] < 0, 'Edad'] = np.nan 

# Mayor IMC y Edad = Más días de estada (simulación GRD)
df_hospital['Dias_Estada'] = np.ceil(df_hospital['Dias_Estada'])
df_hospital['Dias_Estada'] = df_hospital['Dias_Estada'] + (df_hospital['Edad']/20) + (df_hospital['IMC']/10)
df_hospital['Dias_Estada'] = np.ceil(df_hospital['Dias_Estada']) # Se deja como float por los NaN

# Costo = Costo base + (dias de estada * 80.000)
df_hospital['Costo_Atencion_CLP'] = df_hospital['Costo_Atencion_CLP'] + (df_hospital['Dias_Estada'] * 80000)

# Introduciendo valores nulos (NaN) aleatorios en IMC y Presion
idx_nulos_imc = np.random.choice(df_hospital.index, size=25, replace=False)
df_hospital.loc[idx_nulos_imc, 'IMC'] = np.nan

idx_nulos_presion = np.random.choice(df_hospital.index, size=40, replace=False)
df_hospital.loc[idx_nulos_presion, 'Presion_Sistolica'] = np.nan
# ---------------------------------------------------------

print("Datos cargados con éxito.")
df_hospital.head()

## 2. Ejercicios Nivel Control

### Ejercicio 1: Diagnóstico de Datos y Limpieza (20 pts)
1. Muestra cuántos valores nulos (NaN) hay por cada columna.
2. Dado que en salud no podemos inventar datos clínicos sensibles, elimina **todas** las filas que contengan algún valor nulo en las columnas `Edad` o `IMC`.
3. Para la `Presion_Sistolica`, rellena los valores nulos restantes con la **mediana** de dicha columna.
4. Muestra nuevamente el conteo de nulos para comprobar que el DataFrame está limpio.

In [ ]:
# Paso 1: Conteo de nulos usando isnull() y sum()
print("Nulos antes de la limpieza:")
print(df_hospital.isnull().sum())


In [ ]:
# Paso 2: Eliminar filas con nulos específicos en 'Edad' o 'IMC'
# dropna() con subset permite elegir qué columnas mirar para borrar la fila completa.
df_hospital = df_hospital.dropna(subset=['Edad', 'IMC'])

# Paso 3: Imputar nulos en 'Presion_Sistolica' con la mediana
# Primero calculamos la mediana excluyendo los nulos existentes
mediana_presion = df_hospital['Presion_Sistolica'].median()
# Luego rellenamos con fillna()
df_hospital['Presion_Sistolica'] = df_hospital['Presion_Sistolica'].fillna(mediana_presion)

# Paso 4: Comprobación
print("\nNulos después de la limpieza:")
print(df_hospital.isnull().sum())

### Ejercicio 2: Creación de Categorías y Agrupación (`groupby`) (30 pts)
1. Crea una nueva columna llamada `Categoria_Edad` utilizando `pd.cut`. Los rangos deben ser:
   - 0 a 39: 'Joven'
   - 40 a 59: 'Adulto'
   - 60 a 120: 'Adulto Mayor'
2. Utiliza el método `groupby` para calcular el **promedio de `Dias_Estada`** y el **promedio de `Costo_Atencion_CLP`**, agrupado por esta nueva `Categoria_Edad`.
3. *Pregunta de interpretación:* ¿Qué grupo etario representa un mayor costo promedio para el hospital? (Responde en un comentario o celda markdown).

In [ ]:
# Paso 1: Creación de categorías con pd.cut
# Los bins definen los límites (0 a 39, de 39 a 59, etc.)
bins_edad = [0, 39, 59, 120]
etiquetas_edad = ['Joven', 'Adulto', 'Adulto Mayor']
df_hospital['Categoria_Edad'] = pd.cut(df_hospital['Edad'], bins=bins_edad, labels=etiquetas_edad)

# Paso 2: Agrupación y cálculo de promedios
# Agrupamos por la nueva categoría y calculamos la media de las columnas solicitadas
resumen_costos = df_hospital.groupby('Categoria_Edad')[['Dias_Estada', 'Costo_Atencion_CLP']].mean()
print(resumen_costos)

# Respuesta 3: Como se observa en el resultado del groupby,
# el grupo 'Adulto Mayor' representa el mayor costo promedio de atención,
# lo cual es consistente con un mayor promedio de días de estada.

### Ejercicio 3: Análisis de Correlación (25 pts)
Queremos entender qué variables clínicas impactan más en los días que un paciente pasa hospitalizado (Dias_Estada).

1. Calcula la matriz de correlación (solo para variables numéricas).
2. Genera un mapa de calor (`sns.heatmap()`) para visualizar estas correlaciones. *Hint: usa `annot=True` y `cmap='coolwarm'` para que se vea profesional.*
3. *Pregunta de interpretación:* Basado en el gráfico, ¿qué variable clínica (Edad, IMC o Presion) tiene la correlación positiva más fuerte con los `Dias_Estada`?

In [ ]:
# Paso 1: Calcular la matriz de correlación.
# Es importante usar numeric_only=True porque tenemos columnas categóricas (Sexo, Categoria_Edad)
matriz_correlacion = df_hospital.corr(numeric_only=True)

# Paso 2: Visualizar con un mapa de calor
plt.figure(figsize=(8, 6))
sns.heatmap(matriz_correlacion, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Matriz de Correlación de Variables Clínicas')
plt.show()

# Respuesta 3: 
# Al observar la fila o columna de 'Dias_Estada', la variable clínica 
# con la correlación positiva más fuerte es la 'Edad' (el color es más rojo/cálido 
# y el número positivo es más cercano a 1 comparado con IMC o Presión).

### Ejercicio 4: Detección de Valores Atípicos Clínicos (25 pts)
En la gestión hospitalaria y GRD, los "casos atípicos" (outliers) en los días de estada son aquellos que consumen excesivos recursos y requieren revisión médica.

1. Crea un `boxplot` de la variable `Dias_Estada` separada por `Sexo`.
2. Utilizando filtros de Pandas, encuentra y muestra a los pacientes (las filas completas) que hayan estado hospitalizados **estrictamente más de 25 días**.

In [ ]:
# Paso 1: Crear el Boxplot para detectar outliers visualmente
plt.figure(figsize=(8, 5))
sns.boxplot(data=df_hospital, x='Sexo', y='Dias_Estada')
plt.title('Distribución de Días de Estada por Sexo (Detección de Outliers)')
plt.show()

# Paso 2: Filtrar usando Pandas para ver el detalle de esos outliers
# Creamos una máscara booleana (condición) y la aplicamos al DataFrame
pacientes_atipicos = df_hospital[df_hospital['Dias_Estada'] > 25]

print(f"\nSe encontraron {len(pacientes_atipicos)} pacientes con más de 25 días de estada:")
display(pacientes_atipicos)